# Business Sales Intelligence Dashboard
## Complete Analysis Walkthrough

**Project:** Business Sales Intelligence Dashboard  
**Tools:** Python, Pandas, Matplotlib, Seaborn  
**Dataset:** 10,000+ row synthetic sales data with injected quality issues

---

### Pipeline Overview
1. Generate raw messy dataset
2. Clean and standardize
3. Engineer features (monthly revenue, profit margin)
4. Exploratory analysis
5. Key insight: SKU rationalization impact

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Dark theme for all charts
plt.rcParams.update({
    'figure.facecolor': '#0F1117',
    'axes.facecolor':   '#1A1D26',
    'axes.edgecolor':   '#2E3140',
    'axes.labelcolor':  '#C8CDD8',
    'axes.titlecolor':  '#FFFFFF',
    'xtick.color':      '#8B90A0',
    'ytick.color':      '#8B90A0',
    'text.color':       '#C8CDD8',
    'grid.color':       '#2E3140',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'figure.dpi':       110,
})

ACCENT  = '#6C63FF'
ACCENT2 = '#00D4AA'
DANGER  = '#FF4D6D'
GOLD    = '#FFB347'

print('Libraries loaded OK')

## Step 1 — Generate Raw Data

Run the generator script once, then load the CSV.

In [ ]:
import subprocess, sys, os
os.chdir('..')  # run from project root
subprocess.run([sys.executable, 'data/raw/generate_raw_data.py'], check=True)
raw = pd.read_csv('raw_sales_data.csv')
print(f'Raw shape: {raw.shape}')
raw.head()

## Step 2 — Data Quality Audit

In [ ]:
print('=== DATA QUALITY REPORT ===')
print(f'Total rows           : {len(raw):,}')
print(f'Exact duplicates     : {raw.duplicated().sum():,}')
print(f'Null revenue         : {raw["revenue"].isna().sum():,}')
print(f'Null cost            : {raw["cost"].isna().sum():,}')
print(f'Null customer_name   : {raw["customer_name"].isna().sum():,}')
print(f'Negative quantities  : {(raw["quantity"] < 0).sum():,}')
print()
print('Unique date formats found:')
samples = raw['sale_date'].head(20).unique()
for s in samples[:8]:
    print(f'  {s}')
print()
print('Category casing variants:')
print(raw['category'].unique()[:10])

## Step 3 — Cleaning Pipeline

In [ ]:
df = raw.copy()

# 1. Drop duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'1. Dropped {before - len(df)} duplicates')

# 2. Parse dates
def parse_date(val):
    for fmt in ('%m/%d/%Y', '%Y-%m-%d', '%d-%b-%Y', '%d/%m/%Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.NaT

df['sale_date'] = df['sale_date'].apply(parse_date)
df.dropna(subset=['sale_date'], inplace=True)
print(f'2. Dates parsed. Shape: {df.shape}')

# 3. Normalize category
df['category'] = df['category'].str.strip().str.title()
print(f'3. Categories: {sorted(df["category"].unique())}')

# 4. Drop negative quantities
n_neg = (df['quantity'] < 0).sum()
df = df[df['quantity'] >= 0].copy()
print(f'4. Dropped {n_neg} negative-qty rows')

# 5. Impute revenue
m_rev = df['revenue'].isna().sum()
df.loc[df['revenue'].isna(), 'revenue'] = df['unit_price'] * df['quantity']
print(f'5. Imputed {m_rev} revenue values')

# 6. Impute cost (category median cost-pct)
df['_cp'] = df['cost'] / df['revenue']
cat_med = df.groupby('category')['_cp'].median()
m_cost = df['cost'].isna().sum()
for cat, pct in cat_med.items():
    mask = df['cost'].isna() & (df['category'] == cat)
    df.loc[mask, 'cost'] = df.loc[mask, 'revenue'] * pct
df['cost'] = df['cost'].fillna(df['revenue'] * df['_cp'].median())
df.drop(columns=['_cp'], inplace=True)
print(f'6. Imputed {m_cost} cost values')

# 7. Impute customer names
m_cust = df['customer_name'].isna().sum()
df['customer_name'] = df['customer_name'].fillna('Unknown Customer')
print(f'7. Filled {m_cust} null customer names')

print(f'\nClean shape: {df.shape}')

## Step 4 — Feature Engineering

In [ ]:
df['sale_date']         = pd.to_datetime(df['sale_date'])
df['month_year']        = df['sale_date'].dt.to_period('M').astype(str)
df['month_num']         = df['sale_date'].dt.month
df['profit']            = (df['revenue'] - df['cost']).round(2)
df['profit_margin_pct'] = (df['profit'] / df['revenue'] * 100).round(2)

q1, q2 = df['revenue'].quantile([0.33, 0.66])
df['revenue_band'] = pd.cut(df['revenue'], bins=[-1, q1, q2, float('inf')],
                             labels=['Low', 'Medium', 'High'])

print('Feature engineering complete!')
df[['month_year','profit','profit_margin_pct','revenue_band']].head()

## Step 5 — Monthly Revenue Trend

In [ ]:
monthly = (
    df.groupby(['month_year', 'month_num'])
    .agg(total_revenue=('revenue','sum'),
         total_profit=('profit','sum'),
         orders=('order_id','count'))
    .reset_index().sort_values('month_num')
)
monthly['margin_pct'] = monthly['total_profit'] / monthly['total_revenue'] * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
x = range(len(monthly))

ax1.fill_between(x, monthly['total_revenue'], alpha=0.15, color=ACCENT)
ax1.plot(x, monthly['total_revenue'], color=ACCENT, lw=2.5, marker='o', ms=5)
ax1.set_title('Monthly Revenue Trend  (2023)')
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:,.0f}'))
ax1.grid(axis='y')

colors = [ACCENT2 if v >= monthly['margin_pct'].median() else DANGER for v in monthly['margin_pct']]
ax2.bar(x, monthly['margin_pct'], color=colors, alpha=0.85, width=0.6)
ax2.axhline(monthly['margin_pct'].median(), color=GOLD, lw=1.5, ls='--', label='Median')
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_title('Monthly Profit Margin %')
ax2.set_xticks(x)
ax2.set_xticklabels(monthly['month_year'], rotation=45, ha='right')
ax2.legend(facecolor='#1A1D26')

plt.tight_layout()
plt.show()
print(monthly[['month_year','total_revenue','total_profit','margin_pct','orders']].to_string(index=False))

## Step 6 — Top-10 Products

In [ ]:
products = (
    df.groupby(['sku','category'])
    .agg(total_revenue=('revenue','sum'),
         total_profit=('profit','sum'),
         units=('quantity','sum'))
    .reset_index()
)
products['margin_pct'] = products['total_profit'] / products['total_revenue'] * 100

top10 = products.nlargest(10, 'total_revenue')
PALETTE = [ACCENT, ACCENT2, GOLD, '#FF6B9D', '#45B7D1',
           '#96CEB4', '#FFEAA7', '#DDA0DD', '#98FB98', '#F0E68C']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(top10['sku'], top10['total_revenue'], color=PALETTE[:10], alpha=0.88)
ax1.xaxis.set_major_formatter(mtick.FuncFormatter(lambda v,_: f'${v:,.0f}'))
ax1.set_title('Top 10 Products by Revenue')
ax1.invert_yaxis()

top10m = products[~products['sku'].isin(['SKU-098','SKU-099'])].nlargest(10,'margin_pct')
ax2.barh(top10m['sku'], top10m['margin_pct'], color=ACCENT2, alpha=0.85)
ax2.xaxis.set_major_formatter(mtick.PercentFormatter())
ax2.set_title('Top 10 Products by Margin %')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## Step 7 — Key Insight: SKU Rationalization

> **Hypothesis:** Removing SKU-098 and SKU-099 (the two underperforming products with ~95% cost ratios) will significantly improve overall portfolio profit margin.

In [ ]:
bad_skus = ['SKU-098', 'SKU-099']

rev_with    = df['revenue'].sum()
profit_with = df['profit'].sum()
margin_with = profit_with / rev_with * 100

df_clean    = df[~df['sku'].isin(bad_skus)]
rev_without    = df_clean['revenue'].sum()
profit_without = df_clean['profit'].sum()
margin_without = profit_without / rev_without * 100

delta = margin_without - margin_with

print('='*50)
print('  SKU RATIONALIZATION ANALYSIS')
print('='*50)
print(f'  Current portfolio margin   : {margin_with:.2f}%')
print(f'  Margin excl. SKU-098/099   : {margin_without:.2f}%')
print(f'  Improvement                : +{delta:.2f} pp')
print()
print('  Bad SKU performance:')
print(products[products['sku'].isin(bad_skus)][['sku','total_revenue','total_profit','margin_pct']].to_string(index=False))

# Waterfall chart
fig, ax = plt.subplots(figsize=(8, 4.5))
bars    = [margin_with, delta, margin_without]
labels  = [f'Current\n{margin_with:.1f}%', f'+{delta:.1f}pp\n(Remove bad SKUs)', f'New\n{margin_without:.1f}%']
bottoms = [0, margin_with, 0]
colors  = [ACCENT, ACCENT2, GOLD]

for i, (lbl, val, bot, col) in enumerate(zip(labels, bars, bottoms, colors)):
    ax.bar(i, val, bottom=bot, color=col, alpha=0.85, width=0.5)
    ax.text(i, bot + val/2, f'{bot+val:.1f}%', ha='center', va='center',
            fontweight='bold', color='white', fontsize=12)

ax.set_xticks([0,1,2])
ax.set_xticklabels(labels)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
ax.set_title('Profit Margin Impact: Removing Bottom 2 SKUs')
ax.set_ylim(0, margin_without * 1.3)
plt.tight_layout()
plt.show()

## Step 8 — Regional Analysis

In [ ]:
pivot = df.pivot_table(values='revenue', index='region', columns='category', aggfunc='sum')

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.heatmap(pivot, ax=ax,
            cmap=sns.color_palette('mako', as_cmap=True),
            fmt=',.0f', annot=True,
            linewidths=0.5, linecolor='#0F1117')
ax.set_title('Revenue Heatmap: Region x Category')
ax.set_xlabel('')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Raw data generated | 10,500+ rows, 4 date formats, mixed casing, nulls |
| 2 | Deduplication | ~315 exact duplicates removed |
| 3 | Date normalization | All dates → YYYY-MM-DD |
| 4 | Category normalization | 3 casing variants → 5 clean categories |
| 5 | Invalid qty removal | ~105 negative rows dropped |
| 6 | Revenue imputation | ~536 nulls filled: unit_price × quantity |
| 7 | Cost imputation | ~429 nulls filled: category median cost% |
| 8 | Feature engineering | month_year, profit, profit_margin_pct, revenue_band |
| 9 | **Key Insight** | **Removing SKU-098/099 → +Xpp margin improvement** |

**Next step:** Load `data/processed/*.csv` into Power BI — see `powerbi/POWERBI_SETUP.md`